# BTS Flight Delay Prediction - Data Preparation

This notebook prepares the BTS flight dataset for machine learning based on the findings from the Data Understanding phase.
The preparation process includes handling missing and unsuitable records, converting variables into appropriate formats
selecting relevant features, transforming categorical variables and preparing the final dataset for model training.

## 1. Loading the Dataset

The original BTS dataset is loaded again so that the data preparation process starts from the unmodified source data.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

In [3]:
bts_data = pd.read_csv("T_ONTIME_REPORTING.csv")
bts_data.shape

(539747, 20)

## 2. Handling the Target Variable

The target variable for the prediction model is `DEP_DEL15`, which indicates whether a flight departed at least 15 minutes late.

During Data Understanding, 15,923 missing values were found in `DEP_DEL15` and all of these records belonged to cancelled flights. These records need to be handled before the dataset can be used for machine learning.

In [4]:
prepared_data = bts_data.dropna(subset=["DEP_DEL15"])

In [5]:
prepared_data.shape

(523824, 20)

In [6]:
prepared_data["DEP_DEL15"].isnull().sum()

np.int64(0)

With this code, I removed all the missing values from DEP_DEL15.

### Handling Cancelled Flights

Cancelled flights are considered separately because the goal of the model is to predict departure delays for flights that operate. After removing records with a missing target value, the remaining data is checked for cancelled flights.

In [7]:
prepared_data["CANCELLED"].value_counts()

CANCELLED
0.0    523435
1.0       389
Name: count, dtype: int64

### Removing Remaining Cancelled Flights

After removing records with a missing `DEP_DEL15` value, 389 cancelled flights remain in the dataset because they have a
recorded delay classification.

Since the goal of the model is to predict departure delays for flights that operate, cancelled flights are excluded from the
modelling dataset. This prevents the model from learning from flights that did not complete the normal departure process.

In [8]:
prepared_data = prepared_data[prepared_data["CANCELLED"] == 0]

In [9]:
prepared_data["CANCELLED"].value_counts()

CANCELLED
0.0    523435
Name: count, dtype: int64

## 3. Feature Selection

The dataset contains several variables that are not suitable or necessary for the first version of the flight delay prediction
model. These variables are removed before further preparation.

The following variables are excluded:

- `YEAR` and `MONTH` because all records are from January 2025, so these variables do not vary.
- `FL_DATE` because the relevant date information is already represented by `DAY_OF_MONTH` and `DAY_OF_WEEK`.
- `TAIL_NUM` and `OP_CARRIER_FL_NUM` because they mainly act as identifiers and may cause the model to learn patterns specific
to individual aircraft or flights.
- `ORIGIN_CITY_NAME` because the origin airport is already represented by `ORIGIN`.
- `CANCELLED` and `CANCELLATION_CODE` because cancelled flights have already been removed.
- `DIVERTED` because diversion occurs after departure and would not be known when making the prediction.

In [10]:
prepared_data = prepared_data.drop(columns=[
    "YEAR",
    "MONTH",
    "FL_DATE",
    "TAIL_NUM",
    "OP_CARRIER_FL_NUM",
    "ORIGIN_CITY_NAME",
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED"
])

In [11]:
prepared_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 523435 entries, 0 to 539746
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   DAY_OF_MONTH       523435 non-null  int64  
 1   DAY_OF_WEEK        523435 non-null  int64  
 2   OP_UNIQUE_CARRIER  523435 non-null  object 
 3   ORIGIN             523435 non-null  object 
 4   DEST               523435 non-null  object 
 5   CRS_DEP_TIME       523435 non-null  int64  
 6   DEP_DEL15          523435 non-null  float64
 7   DEP_TIME_BLK       523435 non-null  object 
 8   CRS_ARR_TIME       523435 non-null  int64  
 9   CRS_ELAPSED_TIME   523435 non-null  float64
 10  DISTANCE           523435 non-null  float64
dtypes: float64(3), int64(4), object(4)
memory usage: 47.9+ MB


## 4. Transforming Scheduled Time Variables

`CRS_DEP_TIME` and `CRS_ARR_TIME` are stored as integers in HHMM format. Although they are numerical values, they represent clock times rather than normal continuous numbers.

To make these variables more suitable for machine learning, the hour and minute of the scheduled departure and arrival are extracted into separate numerical variables. This preserves the precise scheduled time while representing it in a format that can be used by the model.

After the transformation, the original `CRS_DEP_TIME` and `CRS_ARR_TIME` variables are removed because the same information is now represented by the newly created hour and minute features.

In [12]:
prepared_data["CRS_DEP_HOUR"] = prepared_data["CRS_DEP_TIME"] // 100

In [13]:
prepared_data["CRS_DEP_MINUTE"] = prepared_data["CRS_DEP_TIME"] % 100

In [14]:
prepared_data["CRS_ARR_HOUR"] = prepared_data["CRS_ARR_TIME"] // 100

In [15]:
prepared_data["CRS_ARR_MINUTE"] = prepared_data["CRS_ARR_TIME"] % 100

In [16]:
prepared_data[
    ["CRS_DEP_TIME", "CRS_DEP_HOUR", "CRS_DEP_MINUTE"]
].head(10)

,CRS_DEP_TIME,CRS_DEP_HOUR,CRS_DEP_MINUTE
0,1030,10,30
1,600,6,0
2,819,8,19
3,2100,21,0
4,801,8,1
5,1130,11,30
6,1104,11,4
7,1259,12,59
8,746,7,46
9,2025,20,25


In [17]:
prepared_data[
    ["CRS_ARR_TIME", "CRS_ARR_HOUR", "CRS_ARR_MINUTE"]
].head(10)

,CRS_ARR_TIME,CRS_ARR_HOUR,CRS_ARR_MINUTE
0,1912,19,12
1,940,9,40
2,1206,12,6
3,29,0,29
4,1140,11,40
5,2001,20,1
6,1212,12,12
7,1549,15,49
8,1018,10,18
9,1,0,1


In [18]:
prepared_data = prepared_data.drop(columns=[
    "CRS_DEP_TIME",
    "CRS_ARR_TIME"
])

In [19]:
prepared_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 523435 entries, 0 to 539746
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   DAY_OF_MONTH       523435 non-null  int64  
 1   DAY_OF_WEEK        523435 non-null  int64  
 2   OP_UNIQUE_CARRIER  523435 non-null  object 
 3   ORIGIN             523435 non-null  object 
 4   DEST               523435 non-null  object 
 5   DEP_DEL15          523435 non-null  float64
 6   DEP_TIME_BLK       523435 non-null  object 
 7   CRS_ELAPSED_TIME   523435 non-null  float64
 8   DISTANCE           523435 non-null  float64
 9   CRS_DEP_HOUR       523435 non-null  int64  
 10  CRS_DEP_MINUTE     523435 non-null  int64  
 11  CRS_ARR_HOUR       523435 non-null  int64  
 12  CRS_ARR_MINUTE     523435 non-null  int64  
dtypes: float64(3), int64(6), object(4)
memory usage: 55.9+ MB


## 5. Separating Features and Target

The prepared dataset is separated into features and the target variable before model training.

The target variable is `DEP_DEL15`, which indicates whether a flight was delayed by at least 15 minutes. All remaining variables are used as potential input features for the prediction model.

In [20]:
y = prepared_data["DEP_DEL15"]

X = prepared_data.drop(columns=["DEP_DEL15"])

In [21]:
X.head()

,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,ORIGIN,DEST,DEP_TIME_BLK,CRS_ELAPSED_TIME,DISTANCE,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE
0,1,3,AA,SFO,JFK,1000-1059,342.0,2586.0,10,30,19,12
1,1,3,AA,JFK,SFO,0600-0659,400.0,2586.0,6,0,9,40
2,1,3,AA,SAT,CLT,0800-0859,167.0,1095.0,8,19,12,6
3,1,3,AA,JFK,LAX,2100-2159,389.0,2475.0,21,0,0,29
4,1,3,AA,BOS,LAX,0800-0859,399.0,2611.0,8,1,11,40


In [22]:
y.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: DEP_DEL15, dtype: float64

## 6. Splitting the Training and Test Data

The dataset is divided into training and test data so that the model can be evaluated on flights that were not used during training.

Because flight delay prediction is time-dependent, the data is split chronologically rather than randomly. Flights from the earlier part of January are used for training, while flights from the final part of the month are kept for testing. This better represents a real prediction scenario, where historical flight data is used to predict delays for future flights.

In [23]:
train_mask = X["DAY_OF_MONTH"] <= 24

X_train = X[train_mask]
X_test = X[~train_mask]

y_train = y[train_mask]
y_test = y[~train_mask]

In [24]:
print("X train:", X_train.shape)
print("X test:", X_test.shape)
print("y train:", y_train.shape)
print("y test:", y_test.shape)

X train: (404662, 12)
X test: (118773, 12)
y train: (404662,)
y test: (118773,)


In [25]:
print(X_train["DAY_OF_MONTH"].min())
print(X_train["DAY_OF_MONTH"].max())

1
24


In [26]:
print(X_test["DAY_OF_MONTH"].min())
print(X_test["DAY_OF_MONTH"].max())

25
31


The data was split chronologically using `DAY_OF_MONTH`. Flights from January 1–24 were assigned to the training set, while flights from January 25–31 were assigned to the test set.

This resulted in 404,662 training flights and 118,773 test flights. The number of rows in `X` and `y` was checked to confirm that the features and target remained correctly aligned. The minimum and maximum days were also checked to confirm that the chronological split was performed correctly.

## 7. Encoding Categorical Variables

The remaining categorical variables are `OP_UNIQUE_CARRIER`, `ORIGIN`, `DEST` and `DEP_TIME_BLK`.

These variables contain categories rather than numerical measurements. One-hot encoding will be used to convert each category into a numerical representation without introducing an artificial order between categories.

The encoder is fitted using only the training data and then applied to the test data to prevent information from the test set from influencing the preparation of the training data.

In [27]:
categorical_features = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "DEP_TIME_BLK"
]
encoder = OneHotEncoder(handle_unknown="ignore")

In [28]:
X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

In [29]:
X_train_encoded.shape

(404662, 691)

In [30]:
X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

In [31]:
X_test_encoded.shape

(118773, 691)

### Encoding Results

The categorical variables were one-hot encoded using an encoder fitted only on the training data. The same encoder was then used to transform the test data.

The four original categorical variables were converted into 691 numerical features. The training data contains 404,662 rows and the test data contains 118,773 rows, while both contain the same 691 encoded features.

Fitting the encoder only on the training data helps prevent information from the test data from influencing the preparation process.

In [32]:
numerical_features = [
    column for column in X_train.columns
    if column not in categorical_features
]
print(numerical_features)

['DAY_OF_MONTH', 'DAY_OF_WEEK', 'CRS_ELAPSED_TIME', 'DISTANCE', 'CRS_DEP_HOUR', 'CRS_DEP_MINUTE', 'CRS_ARR_HOUR', 'CRS_ARR_MINUTE']


In [33]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Learn scaling values from training data
X_train_scaled = scaler.fit_transform(
    X_train[numerical_features]
)

# Apply the same scaling to testing data
X_test_scaled = scaler.transform(
    X_test[numerical_features]
)

In [34]:
# Combine encoded features with scaled numerical features
X_train_final = hstack([
    X_train_encoded,
    X_train_scaled
]).tocsr()

X_test_final = hstack([
    X_test_encoded,
    X_test_scaled
]).tocsr()

In [35]:
print("Training:", X_train_final.shape)
print("Testing:", X_test_final.shape)

Training: (404662, 699)
Testing: (118773, 699)


### Combining Encoded and Numerical Features

The one-hot encoded categorical features were combined with the eight existing numerical features.

This resulted in 699 input features for both datasets. The final training set contains 404,662 flights and the test set contains 118,773 flights. Having the same number of features in both datasets ensures that they have a consistent structure for machine learning.

## 8. Final Data Preparation Checks

Before using the prepared data for machine learning, final checks are performed to confirm that the training and test datasets have the expected structure and that the target variable is correctly aligned with the input features.

In [36]:
print("X train:", X_train_final.shape)
print("y train:", y_train.shape)

print("X test:", X_test_final.shape)
print("y test:", y_test.shape)

X train: (404662, 699)
y train: (404662,)
X test: (118773, 699)
y test: (118773,)


In [37]:
print("Missing values in training:", np.isnan(X_train_final.data).sum())
print("Missing values in testing:", np.isnan(X_test_final.data).sum())

Missing values in training: 0
Missing values in testing: 0


In [38]:
print("Training target:")
print(y_train.value_counts())

print("Test target:")
print(y_test.value_counts())

Training target:
DEP_DEL15
0.0    322839
1.0     81823
Name: count, dtype: int64
Test target:
DEP_DEL15
0.0    105164
1.0     13609
Name: count, dtype: int64


### Final Check Results

The final training and test datasets have matching numbers of rows between the input features and target variables. Both datasets contain 699 input features and no missing values were found.

The target variable contains both classes (`0` = not delayed and `1` = delayed). The training set has a higher proportion of delayed flights than the test set. This difference is expected because the data was split chronologically rather than randomly, meaning the test set represents a later period in January with a different observed delay rate.

The prepared datasets are therefore ready to be used for machine learning.

## 9. Data Preparation Conclusion

The BTS dataset was prepared for machine learning based on the findings from the Data Understanding phase.

Flights with a missing target value and cancelled flights were removed. Variables that were constant, redundant, identifier-based or unavailable at prediction time were also excluded.

The scheduled departure and arrival times were transformed into separate hour and minute features. The dataset was then separated into input features (`X`) and the target variable (`DEP_DEL15`).

To simulate predicting future flights, the data was split chronologically. Flights from January 1–24 were used for training and flights from January 25–31 were used for testing.

Categorical variables for airline, origin airport, destination airport and departure time block were one-hot encoded. The encoder was fitted only on the training data and then applied to the test data. These encoded features were combined with the numerical features, resulting in 699 final input features.

The final datasets contain 404,662 training flights and 118,773 test flights. No missing values remain in the prepared features, and both datasets have the same feature structure.

The data is now ready for the modelling phase.


## 10. Saving Prepared Data

The prepared training and testing datasets are saved
so they can be loaded into the modelling notebook
without repeating the preparation process.

In [39]:
from scipy.sparse import save_npz
import numpy as np
from pathlib import Path

# create a folder for the prepared data
output_folder = Path("prepared_data")
output_folder.mkdir(exist_ok=True)

# save input features
save_npz(output_folder / "X_train.npz", X_train_final)
save_npz(output_folder / "X_test.npz", X_test_final)

# save target values
np.save(output_folder / "y_train.npy", y_train.to_numpy())
np.save(output_folder / "y_test.npy", y_test.to_numpy())

print("Prepared datasets saved successfully")


Prepared datasets saved successfully


In [40]:
print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

print("\nSaved files:")
for file in output_folder.iterdir():
    print(file.name)


Training shape: (404662, 699)
Testing shape: (118773, 699)

Saved files:
X_test.npz
X_train.npz
y_test_final_v2.npy
y_train.npy
X_train_final_v2.npz
y_test.npy
X_test_final_v2.npz
y_train_final_v2.npy


## 11. Improvements After Feedback from Martijn

After completing Iteration 0, I discussed my k-NN results with Martijn. One of the main problems was that my model had difficulties predicting delayed flights.

Martijn suggested looking into class balancing and trying a random train/test split instead of splitting the data by date.

In Iteration 0, I used the first 24 days of January for training and the last 7 days for testing. For Iteration 1, I want to try a random split and keep approximately the same percentage of delayed flights in both datasets.

I will keep my original approach to show what I did in Iteration 0 and what I changed after receiving feedback.


In [41]:
from sklearn.model_selection import train_test_split

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [42]:
print (X_train_v2.shape)
print (X_test_v2.shape)

(418748, 12)
(104687, 12)


In [43]:
print (y_train_v2.value_counts(normalize=True))
print(y_test_v2.value_counts(normalize=True))

DEP_DEL15
0.0    0.81768
1.0    0.18232
Name: proportion, dtype: float64
DEP_DEL15
0.0    0.817685
1.0    0.182315
Name: proportion, dtype: float64



### Random Train-Test Split

I used `train_test_split()` to randomly divide the dataset into 80% training data and 20% testing data.

I used `stratify=y` to preserve the original proportion of delayed and on-time flights in both datasets.

The results show approximately 81.77% on-time flights and 18.23% delayed flights in both sets.

This split is different from Iteration 0, where I separated the data chronologically. A chronological split is more realistic for predicting future flights, while the random split allows me to experiment with a different evaluation approach.




### One-Hot Encoding

Since the dataset contains categorical features such as airline, origin airport, destination airport and departure time block, I applied One-Hot Encoding to convert these categories into numerical features.

I fitted a new encoder on the training data and used the same encoder to transform the testing data. This prevents data leakage and ensures both datasets have the same feature structure.

The encoding resulted in 691 categorical features.


In [44]:
categorical_features = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "DEP_TIME_BLK"
]
encoder_v2 = OneHotEncoder(handle_unknown="ignore")
X_train_encoded_v2 = encoder_v2.fit_transform(
    X_train_v2[categorical_features]
)
print (X_train_encoded_v2.shape)

X_test_encoded_v2 = encoder_v2.transform(
    X_test_v2[categorical_features]
)
print (X_test_encoded_v2.shape)

(418748, 691)
(104687, 691)




### Combining Features

After encoding the categorical features, I combined them with the numerical features to create the final datasets for Random Forest.

Unlike k-NN, Random Forest does not require feature scaling because it uses decision trees rather than distance calculations.

I used `hstack()` to combine the features into sparse matrices, keeping the data more memory-efficient.


In [45]:
from scipy.sparse import hstack

# combining encoded categorical features with numerical features
X_train_final_v2 = hstack([
    X_train_encoded_v2,
    X_train_v2[numerical_features].to_numpy()
]).tocsr()

X_test_final_v2 = hstack([
    X_test_encoded_v2,
    X_test_v2[numerical_features].to_numpy()
]).tocsr()

# checking the final shapes
print(X_train_final_v2.shape)
print(X_test_final_v2.shape)

(418748, 699)
(104687, 699)



### Saving the Prepared Data

After completing the preprocessing, I obtained 699 features for both the training and testing datasets.

I saved the new datasets separately from Iteration 0 to preserve my previous work.

These datasets will be used to train and evaluate the Random Forest model.


In [46]:
from scipy.sparse import save_npz
import numpy as np

output_folder = Path("prepared_data")

# saving training and testing features
save_npz(output_folder / "X_train_final_v2.npz", X_train_final_v2)
save_npz(output_folder / "X_test_final_v2.npz", X_test_final_v2)

# saving training and testing labels
np.save(output_folder / "y_train_final_v2.npy", y_train_v2.to_numpy())
np.save(output_folder / "y_test_final_v2.npy", y_test_v2.to_numpy())




### Saving Feature Names

After applying One-Hot Encoding, the dataset contains 699 features. To investigate which features are most relevant for predicting flight delays, I need to identify the individual features.

I saved the names of the encoded categorical features together with the numerical features. This will allow me to use the correct feature names when creating a correlation heatmap and analysing feature importance in the modelling notebook.

In [47]:

# getting the names of the one-hot encoded features
encoded_feature_names = encoder_v2.get_feature_names_out(categorical_features)

# combining encoded and numerical feature names
feature_names_v2 = np.concatenate([
    encoded_feature_names,
    numerical_features
])

# saving feature names for the modelling notebook
np.save(output_folder / "feature_names_v2.npy", feature_names_v2)

# checking the number of features
print("Total features:", len(feature_names_v2))
print("\nFeature names:")
print(feature_names_v2)


Total features: 699

Feature names:
['OP_UNIQUE_CARRIER_AA' 'OP_UNIQUE_CARRIER_AS' 'OP_UNIQUE_CARRIER_B6'
 'OP_UNIQUE_CARRIER_DL' 'OP_UNIQUE_CARRIER_F9' 'OP_UNIQUE_CARRIER_G4'
 'OP_UNIQUE_CARRIER_HA' 'OP_UNIQUE_CARRIER_MQ' 'OP_UNIQUE_CARRIER_NK'
 'OP_UNIQUE_CARRIER_OH' 'OP_UNIQUE_CARRIER_OO' 'OP_UNIQUE_CARRIER_UA'
 'OP_UNIQUE_CARRIER_WN' 'OP_UNIQUE_CARRIER_YX' 'ORIGIN_ABE' 'ORIGIN_ABI'
 'ORIGIN_ABQ' 'ORIGIN_ABR' 'ORIGIN_ABY' 'ORIGIN_ACT' 'ORIGIN_ACV'
 'ORIGIN_ACY' 'ORIGIN_ADK' 'ORIGIN_ADQ' 'ORIGIN_AEX' 'ORIGIN_AGS'
 'ORIGIN_ALB' 'ORIGIN_AMA' 'ORIGIN_ANC' 'ORIGIN_APN' 'ORIGIN_ASE'
 'ORIGIN_ATL' 'ORIGIN_ATW' 'ORIGIN_AUS' 'ORIGIN_AVL' 'ORIGIN_AVP'
 'ORIGIN_AZA' 'ORIGIN_AZO' 'ORIGIN_BDL' 'ORIGIN_BET' 'ORIGIN_BFF'
 'ORIGIN_BFL' 'ORIGIN_BGR' 'ORIGIN_BHM' 'ORIGIN_BIH' 'ORIGIN_BIL'
 'ORIGIN_BIS' 'ORIGIN_BJI' 'ORIGIN_BLI' 'ORIGIN_BLV' 'ORIGIN_BMI'
 'ORIGIN_BNA' 'ORIGIN_BOI' 'ORIGIN_BOS' 'ORIGIN_BPT' 'ORIGIN_BQN'
 'ORIGIN_BRD' 'ORIGIN_BRO' 'ORIGIN_BRW' 'ORIGIN_BTM' 'ORIGIN_BTR'
 'ORIGIN_BTV' 'OR